# 02 Silver transform

Parsed data → a **modelled, query-ready** layer — typed, deduped, and shaped into a small star schema that gold can aggregate without re-deriving anything.

Three output tables:

- **`dim_cards`** — the modelled card dimension, promoted from `bronze_cards`.
- **`silver_battles`** — already deduped, timestamp parsed to a real `timestamp`, `battle_date` derived for partitioning.
- **`silver_deck_cards`** — the decks **exploded** to one row per card appearance
  (8 cards x 2 sides = 16 rows per battle), **broadcast-joined** to `dim_cards`(ideal for joining a massive fact table and a small dimension table). This is the grain gold aggregates for card pick-rate and win-rate.

**Cleaning vs. validation.** Silver_transform *types and reshapes* but does not *drop* rows. Quarantining bad battles (missing deck, negative trophies, unknown card) is layer 3's job (`silver_quality_checks`), so silver stays a faithful, deterministic projection of bronze and the DQ layer has the full population to measure against.

## Config

In [0]:
from pyspark.sql import functions as F, Window, types as T

CATALOG, SCHEMA = "workspace", "clash"

# Sources (bronze)
BRONZE_BATTLES = f"{CATALOG}.{SCHEMA}.bronze_battles"
BRONZE_CARDS   = f"{CATALOG}.{SCHEMA}.bronze_cards"

# Targets (silver)
DIM_CARDS         = f"{CATALOG}.{SCHEMA}.dim_cards"
SILVER_BATTLES    = f"{CATALOG}.{SCHEMA}.silver_battles"
SILVER_DECK_CARDS = f"{CATALOG}.{SCHEMA}.silver_deck_cards"

## `dim_cards` - dimension table

Deduplicate cards by enforcing one row per `card_id`, and derive an `elixir_cost` column for archetype analysis
(cheap / mid / heavy).

In [0]:
card_w = Window.partitionBy("card_id").orderBy(F.col("bronze_loaded_at").desc())

dim_cards = (spark.table(BRONZE_CARDS)
    .withColumn("rn", F.row_number().over(card_w))
    .filter("rn = 1").drop("rn", "bronze_loaded_at")
    .withColumn("elixir_band", F.when(F.col("elixir_cost") <= 2, "cheap")
                                .when(F.col("elixir_cost") <= 4, "mid")
                                .when(F.col("elixir_cost").isNotNull(), "heavy"))
    .withColumn("silver_built_at", F.current_timestamp()))

dim_cards.write.format("delta").mode("overwrite").saveAsTable(DIM_CARDS)
print(spark.table(DIM_CARDS).count(), "cards in dim_cards")
display(spark.table(DIM_CARDS).orderBy("elixir_cost", "card_id").limit(10))

121 cards in dim_cards


card_id,name,rarity,elixir_cost,max_level,elixir_band,silver_built_at
28000006,Mirror,epic,null,11,null,2026-06-22T00:42:08.762Z
26000010,Skeletons,common,1,16,cheap,2026-06-22T00:42:08.762Z
26000030,Ice Spirit,common,1,16,cheap,2026-06-22T00:42:08.762Z
26000031,Fire Spirit,common,1,16,cheap,2026-06-22T00:42:08.762Z
26000084,Electro Spirit,common,1,16,cheap,2026-06-22T00:42:08.762Z
28000016,Heal Spirit,rare,1,14,cheap,2026-06-22T00:42:08.762Z
26000002,Goblins,common,2,16,cheap,2026-06-22T00:42:08.762Z
26000013,Bomber,common,2,16,cheap,2026-06-22T00:42:08.762Z
26000019,Spear Goblins,common,2,16,cheap,2026-06-22T00:42:08.762Z
26000038,Ice Golem,rare,2,14,cheap,2026-06-22T00:42:08.762Z


## `silver_battles` - typed and partitioned battle fact

Deduplication is already done in bronze_battles, so only typing and deriving here.
- `battle_time` (an ISO string in bronze) becomes a real `timestamp`; `battle_date`
  is derived from it as the **partition key**.
- `crown_diff` enables fine-grained winning analysis for the gold layer

Data is processed **incrementally** and appended to the silver table. Exit if no new data loaded from bronze:

In [0]:
bronze = spark.table(BRONZE_BATTLES)

# Get the incremental part of bronze battles
if spark.catalog.tableExists(SILVER_BATTLES):
    latest_timestamp = spark.table(SILVER_BATTLES).select(F.max("ingested_at")).first()[0]
    delta_bronze = bronze.filter(F.col("ingested_at") > latest_timestamp)
else:
    delta_bronze = bronze

delta_count = delta_bronze.count()
print(delta_count, "rows to process")
if delta_count == 0:
    dbutils.notebook.exit("No new data. Notebook stops at silver_battles.")

In [0]:
# Type cast for time and calculate crown difference
silver_battles = (delta_bronze
    .withColumn("battle_time_ts", F.to_timestamp("battle_time"))
    .withColumn("battle_date", F.to_date("battle_time_ts"))
    .withColumn("crown_diff", F.col("player_crowns") - F.col("opponent_crowns"))
    .withColumn("silver_built_at", F.current_timestamp()))

# Drop the raw timestamp strings
silver_battles = silver_battles.drop("battle_time", "battle_time_raw") \
    .withColumnRenamed("battle_time_ts", "battle_time")

# Create if not exists or append to existing table, and partition by date
(silver_battles.write.format("delta").mode("append")
    .partitionBy("battle_date")
    .saveAsTable(SILVER_BATTLES))

print(silver_battles.count(), "new battles")
print(silver_battles.select('battle_date').distinct().count(), "battle dates")
display(silver_battles.limit(10))

## `silver_deck_cards` — exploded card grain

Each battle carries two decks (player + opponent) of card-id arrays. Here they're
**exploded** to one row per card appearance and **broadcast-joined** to `dim_cards`
for name / rarity / elixir.

Both sides are normalised to the same shape — `side`, the deck owner's `player_tag`,
that side's `crowns`, and a side-specific `won` computed from the crown counts (so the
opponent's win is recorded correctly, not just the subject player's). Gold reads this
table directly: `won` per card row gives card win-rate, row counts give pick-rate.

In [0]:
battles = spark.table(SILVER_BATTLES)

def side_frame(side_label, my, opp):
    """Normalise one side of the battle to card grain. `my`/`opp` are the column
    prefixes ('player' / 'opponent') for the deck owner and their adversary."""
    return (battles.select(
        "battle_id", "battle_date", "battle_time", "arena_id", "is_ladder", "game_mode",
        F.lit(side_label).alias("side"),
        F.col(f"{my}_tag").alias("player_tag"),
        F.col(f"{my}_deck_hash").alias("deck_hash"),
        F.col(f"{my}_crowns").alias("crowns"),
        F.col(f"{opp}_crowns").alias("opponent_crowns"),
        F.col(f"{my}_starting_trophies").alias("starting_trophies"),
        F.col(f"{my}_trophy_change").alias("trophy_change"),
        F.explode(f"{my}_cards").alias("card_id"),
    ))

sides = side_frame("player", "player", "opponent").unionByName(
        side_frame("opponent", "opponent", "player"))

# Side-specific outcome from crown counts (handles the opponent's perspective and draws).
sides = sides.withColumn("won",
    F.when(F.col("crowns").isNull() | F.col("opponent_crowns").isNull(), None)
     .otherwise(F.col("crowns") > F.col("opponent_crowns")))

# Deck size per (battle_id, side) — a window count, useful to the DQ '8 cards' check.
deck_w = Window.partitionBy("battle_id", "side")
sides = sides.withColumn("deck_size", F.count("card_id").over(deck_w))

# Broadcast join: dim_cards is tiny, the fact is large -> ship the dim to every executor
# instead of shuffling the fact. Left join so an unknown card_id survives as null name
# (for the DQ 'every card_id in dim' check) rather than dropping the row.
dims = F.broadcast(spark.table(DIM_CARDS).select(
    "card_id", "name", "rarity", "elixir_cost", "elixir_band"))

silver_deck_cards = (sides.join(dims, on="card_id", how="left")
    .withColumnRenamed("name", "card_name")
    .withColumn("silver_built_at", F.current_timestamp()))

(silver_deck_cards.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("battle_date")
    .saveAsTable(SILVER_DECK_CARDS))

sdc = spark.table(SILVER_DECK_CARDS)
print(sdc.count(), "card-rows =", battles.count(), "battles x ~16")
display(sdc.limit(16))

## Sanity checks

Not the formal DQ layer, this is just a guard to check out transform's work before moving on to the next layer.

In [0]:
# 1. silver_battles is one row per battle_id.
sb = spark.table(SILVER_BATTLES)
assert sb.count() == sb.select("battle_id").distinct().count(), "silver_battles not unique on battle_id"

# 2. Every deck-card row points back to a real battle (no explode orphans).
sdc = spark.table(SILVER_DECK_CARDS)
orphans = sdc.join(sb.select("battle_id"), on="battle_id", how="left_anti").count()
assert orphans == 0, f"{orphans} deck-card rows with no parent battle"

# 3. Card coverage: how many rows failed to resolve to a dim_cards name?
unknown = sdc.filter(F.col("card_name").isNull()).count()
print(f"deck-card rows: {sdc.count()};  unresolved card_id: {unknown}")
print("distinct decks (player side):",
      sdc.filter("side = 'player'").select("deck_hash").distinct().count())

## Done

Silver is modelled: `dim_cards`, `silver_battles` (battle grain, partitioned by `battle_date`), and `silver_deck_cards` (card grain, broadcast-joined to the dim). Next: `silver_quality_checks`.